# Beautiful Soup Tutorial

To pip install beautifulsoup4, run the following command in your terminal:

```bash
pip install beautifulsoup4
```
Beautiful Soup is a Python library for parsing HTML and XML documents. It creates a parse tree for parsed pages that can be used to extract data from HTML, which is useful for web scraping.

In this tutorial, we will learn how to use Beautiful Soup to scrape data from a website. We will be using the requests library to fetch the HTML content of a webpage and then use Beautiful Soup to parse it.

In [ ]:
# import beautifulsoup4

import requests
from bs4 import BeautifulSoup


## Status Codes

When you make a request to a webpage using the requests library, you receive a response object. This response object contains a status code that indicates the result of the request. A status code of 200 means that the request was successful, while a status code of 404 means that the page was not found.

Common status codes include:
- 200: OK
- 301: Moved Permanently
- 302: Found (Temporary Redirect)
- 400: Bad Request
- 401: Unauthorized
- 403: Forbidden
- 404: Not Found
- 500: Internal Server Error
- 503: Service Unavailable
- 504: Gateway Timeout
- 505: HTTP Version Not Supported

In [ ]:
# use requests to get the status code of the page

url = "https://www.binghamton.edu/history/"
response = requests.get(url)
print(f"Status Code: {response.status_code}")

In [ ]:
# Get the HTML content of the page
html_content = response.text
print(html_content)

## Parsing with Beautiful Soup

Now that we have the HTML content of the page, we can use Beautiful Soup to parse it. We can find all the links on the page using the `find_all` method.

Let's get the title of the page from the header of the HTML document. The title is usually found within the `<title>` tag in the `<head>` section of the HTML.

In [ ]:
# Get the title of the page
soup = BeautifulSoup(html_content, 'html.parser')
title = soup.title.string
print(f"Title: {title}")

Now extract the links:

In [ ]:
# Import links from https://www.binghamton.edu/history/
url = "https://www.binghamton.edu/history/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
links = soup.find_all('a')
for link in links:
    print(link.get('href'))


In [ ]:
# Now extract not just the links, but also the text of the links
for link in links:
    href = link.get('href')
    text = link.text
    print(f"Link: {href}, Text: {text}")

Extract the paragraphs:

In [ ]:
# Extract the paragraphs from /graduate page
# let's build off of our base url and add the graduate page to it

gradurl = url + "graduate/"
response = requests.get(gradurl)
soup = BeautifulSoup(response.content, 'html.parser')
paragraphs = soup.find_all('p')
for p in paragraphs:
    print(p.text)

Get all of the images from the base url and put them in a folder called "history_images".

In [ ]:
# Get html images from the base url and save them to a folder called "history_images"
import os
from urllib.parse import urljoin

# Fetch the news story page
url = "https://www.binghamton.edu/news/story/5985/the-birth-of-a-nations-care"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

os.makedirs("history_images", exist_ok=True)

images = soup.find_all('img')
print(f"Found {len(images)} <img> tags.")
for img in images:
    print(img)  # Print the full tag for inspection
    img_url = img.get('src')
    # Try data-src or data-lazy if src is missing
    if not img_url:
        img_url = img.get('data-src') or img.get('data-lazy')
    if img_url:
        full_img_url = urljoin(url, img_url)
        print(f"Downloading: {full_img_url}")
        img_name = os.path.basename(img_url)
        img_path = os.path.join("history_images", img_name)
        try:
            img_data = requests.get(full_img_url).content
            with open(img_path, 'wb') as f:
                f.write(img_data)
        except Exception as e:
            print(f"Failed to download {full_img_url}: {e}")


## Digital Florentine Codex

The Digital Florentine Codex is a project that has digitized the Florentine Codex, a 16th-century ethnographic research study in Mesoamerica by the Spanish Franciscan friar Bernardino de Sahagún. The codex is a valuable resource for understanding the culture, history, and society of the Aztec people. The digital version allows researchers and the public to access and explore this important historical document online.

The Digital Florentine Codex can be accessed at [https://florentinecodex.getty.edu/](https://florentinecodex.getty.edu/).

The manuscript’s artists painted about 2,400 scenes and decorative elements (2,472 in total). This total number includes 1,844 images with narrative content and 628 decorative elements or grotesques distributed irregularly throughout the books. For example, Book 5 contains only 9 images and 3 decorative elements, while Book 11 contains a total of 1,137 “pictorial statements.”

The URL for Book 5 is [https://florentinecodex.getty.edu/book/5](https://florentinecodex.getty.edu/book/5). 

In [ ]:
# Fetch the images from book 5

import requests
import os
# The json module provides functions for parsing JSON data, which is commonly used in web applications to exchange data between the client and server. In this code, it is used to parse the JSON data embedded in the HTML of the page to extract the image URLs.
import json
from bs4 import BeautifulSoup

output_dir = "book5_images"
os.makedirs(output_dir, exist_ok=True)

def get_folio_image_url(book_num, folio):
    url = f"https://florentinecodex.getty.edu/book/{book_num}/folio/{folio}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    script = soup.find('script', id='__NEXT_DATA__')
    if not script:
        print(f"No data for folio {folio}")
        return None
    data = json.loads(script.string)
    files = data.get('props', {}).get('pageProps', {}).get('data', {}).get('files', {})
    return files.get('folio_jpg')

# Example: Download folios 1r to 10v
folios = []
for n in range(1, 11):
    folios.append(f"{n}r")
    folios.append(f"{n}v")

for folio in folios:
    img_url = get_folio_image_url(5, folio)
    if img_url:
        img_path = os.path.join(output_dir, f"book5_{folio}.jpg")
        print(f"Downloading {img_url} -> {img_path}")
        img_data = requests.get(img_url).content
        with open(img_path, "wb") as f:
            f.write(img_data)
    else:
        print(f"Image not found for folio {folio}")

In [ ]:
# Extract the nahuatl text and the nahuatl-to-english translations for book 5 ff 1-10

import requests
import os
import json
from bs4 import BeautifulSoup

output_dir = "book5_nahuatl"
os.makedirs(output_dir, exist_ok=True)

def get_nahuatl_text(book_num, folio):
    url = f"https://florentinecodex.getty.edu/book/{book_num}/folio/{folio}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    script = soup.find('script', id='__NEXT_DATA__')
    if not script:
        print(f"No data for folio {folio}")
        return None
    data = json.loads(script.string)
    texts = data.get('props', {}).get('pageProps', {}).get('data', {}).get('texts', {})
    nahuatl = texts.get('nahuatl_col', [])
    return nahuatl

# Example: Extract Nahuatl text for folios 1r to 10v
folios = []
for n in range(1, 11):
    folios.append(f"{n}r")
    folios.append(f"{n}v")

for folio in folios:
    nahuatl = get_nahuatl_text(5, folio)
    if nahuatl:
        outpath = os.path.join(output_dir, f"book5_{folio}_nahuatl.txt")
        with open(outpath, "w", encoding="utf-8") as f:
            for entry in nahuatl:
                f.write(entry.get("markdown", "") + "\n\n")
        print(f"Saved Nahuatl text for folio {folio}")
    else:
        print(f"Nahuatl text not found for folio {folio}")

## API Usage

An API is 


### Ecology of Crisis API

The Ecology of Crisis database (https://ecocrisis.net) contains historical data about famines, epidemics, climate events, and other crises from around the world. It has a much simpler API structure that's great for learning.

Key features:
- Public endpoints (no authentication needed for reading data)
- Historical events categorized by themes (famine, epidemic, flood, etc.)
- Geographic and temporal data
- Time series analysis

Let's use the requests library in Python to interact with the API and process the data programmatically. This allows us to automate data retrieval, handle responses more effectively, and integrate the API data into our analysis pipelines.

In [ ]:
# Get public statistics from the Ecology of Crisis database
import requests
import json

base_url = "https://ecocrisis.net"

# Get overall statistics
response = requests.get(f"{base_url}/eventinfo/public/stats")

stats = response.json()

print("Database Statistics:")
print(json.dumps(stats, indent=2))

Another endpoint is `event/{id}` which will return detailed information about a specific event. For example, `event/101` will return details about the first event in the database.

In [ ]:
import requests

event_id = 101

event_resource = "/eventinfo/public/event"

url = f"{base_url}{event_resource}/{event_id}"

response = requests.get(url)

print(response.json())


We could write the json to a file:

In [ ]:
# Write json data to a file called event_99.json but use event_id in the filename instead of 99
with open(f"event_{event_id}.json", "w", encoding="utf-8") as f:
    json.dump(event, f, ensure_ascii=False, indent=2)

We could also get a more readable output:

In [ ]:
print(f"\nEvent ID: {event.get('id')}")
print(f"Summary: {event.get('summary')}")
print(f"Start Date: {event.get('start_year')}")
print(f"Quote: {event.get('event_quote')}")
print(f"Sources: {event.get('event_source_info')}")

# print all the themes associated with the event, or "None" if there are no themes. Themes are classified into classes, and classes into domains, so add that in parentheses after the theme.
print("Themes:")
themes = event.get('themes', [])
if themes:
    for theme in themes:
        theme_name = theme.get('name', 'Unknown')
        theme_class = theme.get('eventClass', {}).get('name', 'Unknown')
        theme_domain = theme.get('eventClass', {}).get('domain', {}).get('name', 'Unknown')
        print(f"  - {theme_name} ({theme_class}, {theme_domain})")
else:
    print("  - None")

# print all the locations associated with the event, or "None" if there are no locations
print("Locations:")
for loc in event.get('eventGeoLocationSet', []):
    loc_desc = []
    if loc.get('site'):
        loc_desc.append(f"Site: {loc['site']}")
    if loc.get('municipality'):
        loc_desc.append(f"Municipality: {loc['municipality']}")
    if loc.get('state'):
        loc_desc.append(f"State: {loc['state']}")
    if loc.get('country'):
        loc_desc.append(f"Country: {loc['country']}")
    if loc.get('lat') and loc.get('lon'):
        loc_desc.append(f"Coords: ({loc['lat']}, {loc['lon']})")
    print("    - " + ", ".join(loc_desc) if loc_desc else "    - Unknown")

## Query Specific Year

A different endpoint allows us to query events by year or a range of years.

The endpoint is `/eventinfo/public/events/by-year?startYear=YYYY[&endYear=YYYY]`

Let's query all events from a specific year (1697) to see what crisis events occurred that year.

The equivalent curl command is:
```bash
curl -X GET "https://ecocrisis.net/eventinfo/public/events/by-year?startYear=1697&endYear=1697"
```

Let's begin by fetching the first event from the year 1697 using the requests library in Python.

In [20]:
# Get events for 1697 as json file (first event only)
import requests

base_url = "https://ecocrisis.net"

year_range_resource = "/eventinfo/public/events/by-year"

# We can use params to specify the start and end year for our query. In this case, we want to query events that occurred in the year 1697, so we set both startYear and endYear to 1697.

params = {
    "startYear": 1697,
    "endYear": 1697
}

response = requests.get(f"{base_url}{year_range_resource}", params=params)
events_1697 = response.json()

# Print only the first event in a readable format
print(json.dumps(events_1697[0], indent=2, ensure_ascii=False))

{
  "summary": "Buenas cosechas",
  "last_updated": "2025-04-13T02:26:09.188+00:00",
  "event_quote": "\"Don José Sarmiento de Valladares pariente mi virrey gobernador y capitán general de las provincias de Nueva España y presidente de mi Audiencia Real de México o a la persona, o personas que las gobernare, por despacho 4 de noviembre del año pasado de 97, se os ordenó que en los casos de carestía y falta de granos, dispusieseis contribuyesen los eclesiásticos dueños de hacienda con los de su cosecha para ocurrir a la necesidad pública, y en cumplimiento de ésta mi resolución decís en carta de 4 de abril de 99 se observara así en los casos que se ofrecieren pues por el tiempo presente no se necesita de esta providencia con el gran favor que Dios nuestro señor ha hecho a este reino con abundantes cosechas mediante la protección de san Bernardo a quien se ha invocado para ello, y visto en mi Consejo de las Indias ha parecido deciros cuidéis (si llegare el caso) con todo celo y aplicació

Often, we will want to save the json output to a file for later analysis. We can do this using the `json` library in Python to write the response data to a file.

In [22]:
import json
with open("events_1697.json", "w", encoding="utf-8") as f:
    json.dump(events_1697[:20], f, ensure_ascii=False, indent=2)

Let's fetch again but print out the year and summary information for each event.

In [23]:
# Fetch all events from the API and print only year, quote, and source for those with start_year == 1697

# Loop through the filtered events and print the year, quote, and source for each
for event in events_1697:
    print("Year:", event.get('start_year', 'Unknown'))
    print("Quote:", event.get('event_quote', 'No quote'))
    print("Source:", event.get('event_source_info', 'Unknown'))
    # the * 40 will print a line of 40 dashes to separate each event in the output
    print("-" * 40)

Year: 1697
Quote: "Don José Sarmiento de Valladares pariente mi virrey gobernador y capitán general de las provincias de Nueva España y presidente de mi Audiencia Real de México o a la persona, o personas que las gobernare, por despacho 4 de noviembre del año pasado de 97, se os ordenó que en los casos de carestía y falta de granos, dispusieseis contribuyesen los eclesiásticos dueños de hacienda con los de su cosecha para ocurrir a la necesidad pública, y en cumplimiento de ésta mi resolución decís en carta de 4 de abril de 99 se observara así en los casos que se ofrecieren pues por el tiempo presente no se necesita de esta providencia con el gran favor que Dios nuestro señor ha hecho a este reino con abundantes cosechas mediante la protección de san Bernardo a quien se ha invocado para ello, y visto en mi Consejo de las Indias ha parecido deciros cuidéis (si llegare el caso) con todo celo y aplicación de observar en este punto las órdenes referidas. Don Joseph Sarmiento de Valladares,